In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

[回路への命令の可視化](/guides/visualize-circuits)に加えて、Qiskit の [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) メソッドを使用して回路のスケジューリングを可視化することもできます。この可視化は、例えば量子ビット上のアイドル時間をすばやく発見するのに役立ちます。ただし、このメソッドは動的回路に対して正確な結果を返しません。動的回路のスケジューリングを可視化するには、[Qiskit Runtime サポート](#qr-support) セクションで説明している `draw_circuit_schedule_timing` メソッドを使用してください。

## 例

スケジュールされた回路プログラムを可視化するには、一連の制御引数を指定してこの関数を呼び出します。出力画像の外観のほとんどはスタイルシートで変更できますが、必須ではありません。

### デフォルトのスタイルシートで描画する

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![前のコードセルの出力](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### プログラムデバッグに適したスタイルシートで描画する

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![前のコードセルの出力](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

カスタムのジェネレーター関数またはレイアウト関数を作成し、カスタム関数で既存のスタイルシートを更新することができます。これにより、スケジュールされた回路ドロワーのコードベースを変更することなく、出力画像の外観のほとんどを制御できます。詳細な例については、[`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) の API リファレンスを参照してください。

<span id="qr-support"></span>
## Qiskit Runtime サポート
Qiskit に組み込まれているタイムライン描画機能は静的回路には有用ですが、ブロードキャストやブランチ決定などの暗黙的な操作のために、[動的回路](/guides/classical-feedforward-and-control-flow)のタイミングを正確に反映しない場合があります。動的回路サポートの一環として、Qiskit Runtime はリクエストされた場合にジョブ結果の中に正確な回路タイミング情報を返します。

> **Note:** - これは実験的な機能です。プレビューリリース状態にあるため、変更される可能性があります。
> - この機能は Qiskit Runtime Sampler ジョブにのみ適用されます。
> - 回路の合計時間は「compilation」メタデータに返されますが、これは課金（量子時間）に使用される時間ではありません。

### タイミングデータの取得を有効にする
タイミングデータの取得を有効にするには、プリミティブジョブを実行する際に実験的な `scheduler_timing` フラグを `True` に設定します。